# Atividade Integrada: Ciência de Dados aplicada às Importações Brasileiras e Geopolítica

### Etapa 1: Importação das bibliotecas

In [11]:
import pandas as pd
import sqlite3
import plotly.express as px
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

### Etapa 2: Carregamento dos dados

In [2]:
conn = sqlite3.connect('database.db')
query_tables = "SELECT name FROM sqlite_master WHERE type='table';"
df = pd.read_sql(query_tables, conn)

display(df)
conn.close()

,name
0,co_ncm
1,co_pais
2,co_unid
3,co_urf
4,co_via
5,exp_completa
6,imp_completa
7,ncm_unidade


### Etapa 2.1: Seleção das tabelas 

In [3]:
conn = sqlite3.connect('database.db')
query_importacoes = """SELECT
                        CO_ANO,
                        CO_PAIS,
                        CO_NCM,
                        VL_FOB,
                        KG_LIQUIDO
                        FROM imp_completa"""

df_importacoes = pd.read_sql(query_importacoes, conn)

display(df_importacoes.head())
conn.close()

,CO_ANO,CO_PAIS,CO_NCM,VL_FOB,KG_LIQUIDO
0,1997,160,85369090,7384,833
1,1997,249,85164000,75,6
2,1997,160,82159990,5289,7331
3,1997,23,32041100,255293,11839
4,1997,249,62104000,1044,24


### Etapa 2.2: Selecionando apenas os anos mais recentes

- Aqui eu vou usar os últimos 4 anos como base

In [4]:
conn = sqlite3.connect('database.db')
query_co_ano_recente = "SELECT DISTINCT CO_ANO FROM imp_completa ORDER BY CO_ANO DESC"

df_anos = pd.read_sql(query_co_ano_recente, conn)

print(df_anos['CO_ANO'].to_list())

conn.close()

[2021, 2020, 2019, 2018, 2017, 2016, 2015, 2014, 2013, 2012, 2011, 2010, 2009, 2008, 2007, 2006, 2005, 2004, 2003, 2002, 2001, 2000, 1999, 1998, 1997]


In [5]:
conn = sqlite3.connect('database.db')
query_importacoes = """SELECT
                            CO_ANO,
                            CO_PAIS,
                            CO_NCM,
                            VL_FOB,
                            KG_LIQUIDO
                        FROM imp_completa
                        WHERE CO_ANO > 2017
                        """
df_importacoes = pd.read_sql(query_importacoes, conn)
display(df_importacoes.head())

conn.close()

,CO_ANO,CO_PAIS,CO_NCM,VL_FOB,KG_LIQUIDO
0,2018,386,83089010,51,0
1,2018,87,39023000,1306024,731716
2,2018,493,33021000,506,46
3,2018,399,84122900,37763,4737
4,2018,23,73202010,210681,23429


### Etapa 3: Análise Exploratória (EDA)

a) Top países exportadores para o Brasil

Responda:

    - O Brasil importa mais de países aliados aos EUA?

    Resposta: Sim, apesar da China está em primeiro, quando você olha para a lista de nomes, o Brasil importa mais do alidados dos EUA.

    - Países do Oriente Médio aparecem neste ranking?

    Resposta: Considerando a lista que puxa os top 15, os países do Oriente Médio não aparecem nessa lista.

In [6]:
conn = sqlite3.connect('database.db')
query = """
SELECT
    imp.CO_ANO,
    p.CO_PAIS,
    imp.CO_NCM,
    imp.VL_FOB,
    imp.KG_LIQUIDO
FROM imp_completa AS imp
JOIN co_pais AS p ON imp.CO_PAIS = p.CO_PAIS
WHERE CO_ANO > 2017
"""

df = pd.read_sql(query, conn)
display(df.head())

conn.close()

,CO_ANO,CO_PAIS,CO_NCM,VL_FOB,KG_LIQUIDO
0,2018,386,83089010,51,0
1,2018,87,39023000,1306024,731716
2,2018,493,33021000,506,46
3,2018,399,84122900,37763,4737
4,2018,23,73202010,210681,23429


In [7]:
import sqlite3
import pandas as pd

conn = sqlite3.connect('database.db')

df_dicionario = pd.read_sql("SELECT * FROM co_pais LIMIT 5", conn)
display(df_dicionario)

conn.close()

,CO_PAIS,CO_PAIS_ISON3,CO_PAIS_ISOA3,NO_PAIS,NO_PAIS_ING,NO_PAIS_ESP
0,0,898,ZZZ,Não Definido,Not defined,No definido
1,13,4,AFG,Afeganistão,Afghanistan,Afganistan
2,15,248,ALA,"Aland, Ilhas",Aland Islands,"Alans, Islas"
3,17,8,ALB,Albânia,Albania,Albania
4,20,724,ESP,"Alboran-Perejil, Ilhas","Alboran-Perejil, Islands","Alboran-Perejil, Islas"


In [8]:
conn = sqlite3.connect('database.db')
query = """
SELECT
    imp.CO_ANO,
    p.NO_PAIS,
    imp.VL_FOB
FROM imp_completa AS imp
JOIN co_pais AS p ON imp.CO_PAIS = p.CO_PAIS
WHERE CO_ANO > 2017
"""

df = pd.read_sql(query, conn)

conn.close()

ranking_paises = df.groupby('NO_PAIS')['VL_FOB'].sum().reset_index()
ranking_top15 = ranking_paises.sort_values(by='VL_FOB', ascending=False).head(15)

print("15 Países que o Brasil mais importa (2018 - 2021):")
display(ranking_top15)

15 Países que o Brasil mais importa (2018 - 2021):


,NO_PAIS,VL_FOB
49,China,119884865852
77,Estados Unidos,106202912162
4,Alemanha,34655441983
11,Argentina,33128690354
59,Coreia do Sul,16929606897
153,México,15754257890
118,Japão,15753085491
85,França,15663533124
115,Itália,15438358708
251,Índia,14440294487


### Etapa 3: Análise Exploratória (EDA)

b) Produtos estratégicos afetados pela guerra

Interpretação:
    
- Existe aumento nas importações de energia após o início do conflito?

    Analisando o gráfico abaixo e os dados que estão presentes na internet, o gasto de importação de energia é bem variável, logo depende muito, mas na minha visão não houve aumento.

In [10]:
conn = sqlite3.connect('database.db')

query_energia = """
SELECT
    CO_ANO,
    VL_FOB,
    KG_LIQUIDO
FROM imp_completa
WHERE CO_ANO > 2017
    AND CO_NCM >= 27_000_000
    AND CO_NCM <= 27_999_999
"""

df_energia = pd.read_sql(query_energia, conn)
conn.close()

evolucao_energia = df_energia.groupby('CO_ANO')[['VL_FOB', 'KG_LIQUIDO']].sum().reset_index()

print("Importação de Energia por Ano")
display(evolucao_energia)

fig = px.bar(
    evolucao_energia,
    x='CO_ANO',
    y='VL_FOB',
    title="Gasto do Brasil com Importação de Energia",
    labels={'CO_ANO': 'Ano Econômico', 'VL_FOB': 'Valor Gasto (US$ FOB)'},
    template='plotly_white'
)

fig.add_vline(
    x=2019.5, 
    line_width=3, 
    line_dash="dash", 
    line_color="red", 
    annotation_text="Início da Guerra")
fig.show()

Importação de Energia por Ano


,CO_ANO,VL_FOB,KG_LIQUIDO
0,2018,27872422772,68509873272
1,2019,25465004853,66100304061
2,2020,15582887134,54444986943
3,2021,7723168990,23254410876


### Etapa 4: Modelagem simples

Classificar se um ano econômico foi:
- 0 → Estável
-  1 → Impactado por crise internacional

In [14]:
df_energia['CRISE'] = df_energia['CO_ANO'].apply(lambda ano: 1 if ano >= 2020 else 0)

X = df_energia[['VL_FOB', 'KG_LIQUIDO']]
y = df_energia['CRISE']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

modelo_crise = DecisionTreeClassifier(random_state=42)
modelo_crise.fit(X_train, y_train)

previsoes = modelo_crise.predict(X_test)

acuracia = accuracy_score(y_test, previsoes)

print(f"Acurácia do Modelo: {acuracia * 100:.2f}%\n")

teste_visual = X_test.head().copy()
teste_visual['Cenário Real'] = y_test.head().map({0: 'Estável', 1: 'Crise'})
teste_visual['Previsões da IA'] = previsoes[:5]
teste_visual['Previsões da IA'] = teste_visual['Previsões da IA'].map({0: 'Estável', 1: 'Crise'})

display(teste_visual)

Acurácia do Modelo: 56.41%



,VL_FOB,KG_LIQUIDO,Cenário Real,Previsões da IA
17348,370260,6800000,Crise,Crise
17008,85207,168000,Crise,Estável
18541,3644,740,Crise,Crise
369,5983,1456,Estável,Crise
13102,487,50,Crise,Crise


### Reflexão e Interpretação

1. Quais setores brasileiros são mais sensíveis a conflitos internacionais?

    Agronegócio, Transporte e Logística, Energia, Indústria Química e Farmacêutica.

2. O Brasil é realmente “protegido” da guerra?

    Creio que apesar do Brasil ter grandes riquezas naturais, nenhum país consegue ficar protegido da Guerra, pois pode acontecer de elevarem os preços de produtos, principalmente hoje com a crise de chips, logo, nenhum país está totalmente protegido de uma Guerra.

3. Como a Ciência de Dados pode ajudar governos a prever crises?

    Com base em análise gráfica e estudos de casos como a obtenção de uma extensa base de dados com o objetivo de treinar um modelo para que possa haver uma base de informação confiável sobre o estudo de um determinado cenário de crise.

4. Que outras variáveis poderiam melhorar o modelo?

    Taxas de câmbio, Commodities.